In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(4242)

plt.rcParams.update({
    'text.usetex': True,
    'font.size': 18
})


A = np.diag([1,1,0])
print(f'Norm of A: {np.linalg.norm(A,ord=2)}')

# our method

v = np.random.randn(3)
v /= np.linalg.norm(v)
Av = A@v
normAv = np.linalg.norm(Av)


plt.plot()

for k in range(5000):
    y = np.random.randn(3)
    x = y - np.sum(y*v)*v
    x /= np.linalg.norm(x)
    Ax = A@x
    normAx = np.linalg.norm(Ax)
    a = np.sum(Ax*Av)
    b = normAx**2 - normAv**2
    tau = np.sign(a)*(b/(2*np.abs(a)) + np.sqrt(b**2/(4*a**2)+1))
    v += tau*x
    v /= np.linalg.norm(v)
    Av = A@v
    nA = np.linalg.norm(Av)
    print(f'Iter {k+1} norm est: {nA}')
    if np.abs(v[2])<1e-2:
        plt.plot(v[0],v[1],'ro')


ax = plt.gca()
ax.set_aspect('equal', adjustable='box')
plt.savefig(f'experiment2.pdf')
plt.show()


In [ ]:
# Again but now with stopping criterion
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(4242)
plt.rcParams.update({
    'text.usetex': True,
    'font.size': 18
})


A = np.diag([1,1,0])
print(f'Norm of A: {np.linalg.norm(A,ord=2)}')

# our method

v = np.random.randn(3)
v /= np.linalg.norm(v)
Av = A@v
normAv = np.linalg.norm(Av)

stopTol = 1e-2
stopCount = 0
a_s = []

for k in range(5000):
    y = np.random.randn(3)
    x = y - np.sum(y*v)*v
    x /= np.linalg.norm(x)
    Ax = A@x
    normAx = np.linalg.norm(Ax)
    a = np.sum(Ax*Av)
    a2 = a**2
    a_s.append(a)
    if a2 < stopTol**2:
        N = 20
        for l in range(N):
            y = np.random.randn(3)
            x = y - np.sum(y*v)*v
            x /= np.linalg.norm(x)
            Ax = A@x
            print(f'{l}, {np.sum(Ax*Av)}')
            a2 += np.sum(Ax*Av)**2
        if a2/N < stopTol**2:
            print('Exmpirical mean of a has become small, stop here')
            print(f'Step {k+1: 5d}. Est: {nA:2.4f}, a^2; {a2:2.2e} (step with size {tau:2.2e})')
            break
    b = normAx**2 - normAv**2
    tau = np.sign(a)*(b/(2*np.abs(a)) + np.sqrt(b**2/(4*a**2)+1))
    v += tau*x
    v /= np.linalg.norm(v)
    Av = A@v
    nA = np.linalg.norm(Av)


    
    print(f'Iter {k+1} norm est: {nA}')




In [ ]:
# Next experiment with a single row as matrix:

def runexample(n,step):
    # Make 50 runs of Algorithm 1 with a nx1 matrix with just one nonzero entry and 10*n steps.
    # Shows results and exports plot (with step as subsampling to save memory)
    
    # matrix
    A = np.zeros((1,n))
    A[0,0] = 1.0
    
    # and its norm
    normA = np.linalg.norm(A,ord=2)
    
    print(f'norm(A) = {normA}')
    
    
    maxiter = 10*n  # number of iterations
    check = round(maxiter/1)    # how often do we give output
    
    
    
    numRuns = 50
    color='C0'
    alpha = 0.2
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(3,7), constrained_layout=True)
    
    for run in range(numRuns):
        # initialize with random unit vector
        v = np.random.randn(n)
        v /= np.linalg.norm(v)
        Av = A@v
        normAv = np.linalg.norm(Av)
        
        # list of estimates
        nA = []
        
        # list of as
        a_s = []

        # list of norms on last components of v
        v_last = []
        
        nA.append(normAv)
        for i in range(maxiter):
            # sample x orthogonal to v with unit norm
            x = np.random.randn(n)
            x -= np.dot(x,v)*v
            x /= np.linalg.norm(x)
            Ax = A@x
            normAx = np.linalg.norm(Ax)
            a = np.dot(Ax,Av)
            b = normAx**2 - normAv**2
            tau = np.sign(a)*(b/(2*np.abs(a)) + np.sqrt(b**2/(4*a**2)+1))
            v += tau*x
            v /= np.linalg.norm(v)
            Av = A@v
            normAv = np.linalg.norm(Av)
            nA.append(normAv)
            a_s.append(a)
            v_last.append(np.linalg.norm(v[1:n]))
            
        print(f'Step {i: 5d}. Est: {nA[-1]:2.2f}, (step with size {tau:2.2e})')
            
        print(f'   Reached norm with relative error {100*(normA-nA[-1])/normA:2.2f}%')
        
        axes[0].semilogy(np.arange(1,maxiter+1)[::step],(normA - nA[:-1:step])/normA, color=color, alpha=alpha)
        axes[0].set_xlabel('$k$')
        axes[0].set_ylabel('$(\|A\| - \|Av^k\|)/\|A\|$')

        axes[1].semilogy(np.arange(1,maxiter)[::step],v_last[:-1:step], color=color, alpha=alpha)
        axes[1].set_xlabel('$k$')
        axes[1].set_ylabel('$\|v^k[2:d]\|$')
    #axes[1].legend()
        
    
    plt.savefig(f'experiment2_{n}.pdf')
    plt.show()

runexample(10000,5)